In [ ]:
# ============================================================
# WILDLIFE APPROACH DETECTION
# IR + ENVIRONMENTAL AUDIO
# FFT + FEATURE FUSION + SMOTE + RANDOM FOREST
# COMPLETE NOTEBOOK IN ONE CELL
# ============================================================

!pip install numpy pandas scipy scikit-learn imbalanced-learn matplotlib -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.fft import rfft, rfftfreq

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    accuracy_score,
    confusion_matrix,
    classification_report
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

np.random.seed(42)


# ============================================================
# 1. PARAMETERS
# ============================================================

N_SAMPLES = 1000

IR_SAMPLING_RATE = 10       # 10 Hz
IR_DURATION = 10             # seconds

AUDIO_SAMPLING_RATE = 48000  # 48 kHz
AUDIO_DURATION = 2            # seconds

N_APPROACHING = 450
N_NOT_APPROACHING = 550


# ============================================================
# 2. CREATE LABELS
# ============================================================

y = np.array(
    [1] * N_APPROACHING +
    [0] * N_NOT_APPROACHING
)

np.random.shuffle(y)

print("============================================")
print("WILDLIFE APPROACH DETECTION")
print("============================================")
print("Total samples     :", len(y))
print("Approaching       :", np.sum(y == 1))
print("Not Approaching   :", np.sum(y == 0))


# ============================================================
# 3. GENERATE IR SIGNAL
# ============================================================

def generate_ir_signal(label):

    t = np.arange(
        0,
        IR_DURATION,
        1 / IR_SAMPLING_RATE
    )

    # Background noise
    signal = 0.20 * np.random.randn(len(t))

    if label == 1:

        # Simulated approaching-animal motion
        signal += 1.5 * np.sin(
            2 * np.pi * 0.7 * t
        )

        signal += 0.8 * np.sin(
            2 * np.pi * 2.3 * t
        )

    else:

        # Simulated background movement
        signal += 0.3 * np.sin(
            2 * np.pi * 0.2 * t
        )

    return signal


# ============================================================
# 4. GENERATE AUDIO SIGNAL
# ============================================================

def generate_audio_signal(label):

    t = np.arange(
        0,
        AUDIO_DURATION,
        1 / AUDIO_SAMPLING_RATE
    )

    # Environmental noise
    signal = 0.01 * np.random.randn(len(t))

    if label == 1:

        # Simulated animal-related sound
        signal += 0.15 * np.sin(
            2 * np.pi * 2000 * t
        )

        signal += 0.10 * np.sin(
            2 * np.pi * 5000 * t
        )

        signal += 0.08 * np.sin(
            2 * np.pi * 11000 * t
        )

    else:

        # Simulated environmental sound
        signal += 0.05 * np.sin(
            2 * np.pi * 1000 * t
        )

    return signal


# ============================================================
# 5. FFT
# ============================================================

def calculate_fft(signal, sampling_rate):

    fft_values = rfft(signal)

    power = np.abs(fft_values) ** 2

    frequencies = rfftfreq(
        len(signal),
        d=1 / sampling_rate
    )

    return frequencies, power


# ============================================================
# 6. IR FEATURE EXTRACTION
# ============================================================

def extract_ir_features(signal, resolution):

    frequencies, power = calculate_fft(
        signal,
        IR_SAMPLING_RATE
    )

    max_freq = IR_SAMPLING_RATE / 2

    bins = np.arange(
        0,
        max_freq + resolution,
        resolution
    )

    features = []

    for i in range(len(bins) - 1):

        low = bins[i]
        high = bins[i + 1]

        mask = (
            (frequencies >= low) &
            (frequencies < high)
        )

        if np.any(mask):
            features.append(
                np.mean(power[mask])
            )
        else:
            features.append(0)

    return np.array(features)


# ============================================================
# 7. AUDIO FEATURE EXTRACTION
# ============================================================

def extract_audio_features(signal, resolution):

    frequencies, power = calculate_fft(
        signal,
        AUDIO_SAMPLING_RATE
    )

    max_freq = AUDIO_SAMPLING_RATE / 2

    bins = np.arange(
        0,
        max_freq + resolution,
        resolution
    )

    features = []

    for i in range(len(bins) - 1):

        low = bins[i]
        high = bins[i + 1]

        mask = (
            (frequencies >= low) &
            (frequencies < high)
        )

        if np.any(mask):
            features.append(
                np.mean(power[mask])
            )
        else:
            features.append(0)

    return np.array(features)


# ============================================================
# 8. CREATE FUSED DATASET
# ============================================================

def create_dataset(ir_resolution, audio_resolution):

    X = []

    for label in y:

        # Generate IR
        ir_signal = generate_ir_signal(label)

        # Generate audio
        audio_signal = generate_audio_signal(label)

        # IR FFT features
        ir_features = extract_ir_features(
            ir_signal,
            ir_resolution
        )

        # Audio FFT features
        audio_features = extract_audio_features(
            audio_signal,
            audio_resolution
        )

        # Feature-level fusion
        combined_features = np.concatenate(
            [
                ir_features,
                audio_features
            ]
        )

        X.append(combined_features)

    # IMPORTANT:
    # Return BOTH X and y
    return np.array(X), y.copy()


# ============================================================
# 9. EXAMPLE DATASET
# ============================================================

print("\nCreating example dataset...")

X, y_final = create_dataset(
    ir_resolution=0.1,
    audio_resolution=4000
)

print("Feature matrix shape:", X.shape)
print("Labels shape        :", y_final.shape)


# ============================================================
# 10. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_final,
    test_size=0.20,
    stratify=y_final,
    random_state=42
)

print("\nTraining samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])


# ============================================================
# 11. RANDOM FOREST + SMOTE
# ============================================================

rf_pipeline = Pipeline([

    (
        "smote",
        SMOTE(random_state=42)
    ),

    (
        "rf",
        RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )
    )
])

rf_pipeline.fit(
    X_train,
    y_train
)

rf_probability = rf_pipeline.predict_proba(
    X_test
)[:, 1]

rf_prediction = rf_pipeline.predict(
    X_test
)

rf_auc = roc_auc_score(
    y_test,
    rf_probability
)

rf_accuracy = accuracy_score(
    y_test,
    rf_prediction
)


# ============================================================
# 12. RANDOM FOREST RESULTS
# ============================================================

print("\n============================================")
print("RANDOM FOREST RESULTS")
print("============================================")

print("AUC      :", round(rf_auc, 4))
print("Accuracy :", round(rf_accuracy, 4))

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        rf_prediction
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        rf_prediction,
        target_names=[
            "Not Approaching",
            "Approaching"
        ]
    )
)


# ============================================================
# 13. ROC CURVE
# ============================================================

fpr, tpr, thresholds = roc_curve(
    y_test,
    rf_probability
)

plt.figure(figsize=(7, 5))

plt.plot(
    fpr,
    tpr,
    label=f"Random Forest AUC = {rf_auc:.3f}"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title(
    "ROC Curve - IR + Environmental Audio"
)

plt.legend()
plt.grid()
plt.show()


# ============================================================
# 14. LOGISTIC REGRESSION
# ============================================================

lr_pipeline = Pipeline([

    (
        "smote",
        SMOTE(random_state=42)
    ),

    (
        "scaler",
        StandardScaler()
    ),

    (
        "lr",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

lr_pipeline.fit(
    X_train,
    y_train
)

lr_probability = lr_pipeline.predict_proba(
    X_test
)[:, 1]

lr_prediction = lr_pipeline.predict(
    X_test
)

lr_auc = roc_auc_score(
    y_test,
    lr_probability
)

lr_accuracy = accuracy_score(
    y_test,
    lr_prediction
)

print("\n============================================")
print("LOGISTIC REGRESSION RESULTS")
print("============================================")

print("AUC      :", round(lr_auc, 4))
print("Accuracy :", round(lr_accuracy, 4))


# ============================================================
# 15. PCA + RANDOM FOREST
# ============================================================

pca_pipeline = Pipeline([

    (
        "smote",
        SMOTE(random_state=42)
    ),

    (
        "scaler",
        StandardScaler()
    ),

    (
        "pca",
        PCA(
            n_components=0.90,
            random_state=42
        )
    ),

    (
        "rf",
        RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )
    )
])

pca_pipeline.fit(
    X_train,
    y_train
)

pca_probability = pca_pipeline.predict_proba(
    X_test
)[:, 1]

pca_prediction = pca_pipeline.predict(
    X_test
)

pca_auc = roc_auc_score(
    y_test,
    pca_probability
)

pca_accuracy = accuracy_score(
    y_test,
    pca_prediction
)

print("\n============================================")
print("PCA + RANDOM FOREST RESULTS")
print("============================================")

print("AUC      :", round(pca_auc, 4))
print("Accuracy :", round(pca_accuracy, 4))

print(
    "Original features:",
    X.shape[1]
)

print(
    "PCA features     :",
    pca_pipeline.named_steps["pca"].n_components_
)


# ============================================================
# 16. STRATIFIED 5-FOLD CROSS VALIDATION
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_pipeline = Pipeline([

    (
        "smote",
        SMOTE(random_state=42)
    ),

    (
        "rf",
        RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )
    )
])

cv_scores = cross_val_score(
    cv_pipeline,
    X,
    y_final,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

print("\n============================================")
print("STRATIFIED 5-FOLD CROSS VALIDATION")
print("============================================")

print(
    "Fold AUC scores:",
    np.round(cv_scores, 4)
)

print(
    "Mean AUC:",
    round(cv_scores.mean(), 4)
)

print(
    "Std AUC :",
    round(cv_scores.std(), 4)
)


# ============================================================
# 17. ALL SIX CONFIGURATIONS
# ============================================================

configurations = [

    ("Dataset 1", 0.1, 1000),
    ("Dataset 2", 0.1, 2000),
    ("Dataset 3", 0.1, 4000),

    ("Dataset 4", 1.0, 1000),
    ("Dataset 5", 1.0, 2000),
    ("Dataset 6", 1.0, 4000)
]

results = []


for name, ir_res, audio_res in configurations:

    print(
        f"\nRunning {name} "
        f"(IR={ir_res} Hz, "
        f"Audio={audio_res/1000:.0f} kHz)"
    )

    # Generate fused dataset
    X_config, y_config = create_dataset(
        ir_resolution=ir_res,
        audio_resolution=audio_res
    )

    # Train/test split
    Xtr, Xte, ytr, yte = train_test_split(
        X_config,
        y_config,
        test_size=0.20,
        stratify=y_config,
        random_state=42
    )

    # Random Forest + SMOTE
    model = Pipeline([

        (
            "smote",
            SMOTE(random_state=42)
        ),

        (
            "rf",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    model.fit(
        Xtr,
        ytr
    )

    probability = model.predict_proba(
        Xte
    )[:, 1]

    prediction = model.predict(
        Xte
    )

    auc = roc_auc_score(
        yte,
        probability
    )

    accuracy = accuracy_score(
        yte,
        prediction
    )

    results.append({

        "Dataset": name,

        "IR Resolution (Hz)": ir_res,

        "Audio Resolution (Hz)": audio_res,

        "Features": X_config.shape[1],

        "AUC": auc,

        "Accuracy": accuracy
    })


# ============================================================
# 18. RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(
    results
)

print("\n============================================")
print("ALL SIX CONFIGURATION RESULTS")
print("============================================")

display(
    results_df.round(4)
)


# ============================================================
# 19. AUC COMPARISON
# ============================================================

plt.figure(figsize=(9, 5))

plt.bar(
    results_df["Dataset"],
    results_df["AUC"]
)

plt.xlabel("Dataset Configuration")
plt.ylabel("Test AUC")

plt.title(
    "AUC Comparison - IR + Audio"
)

plt.ylim(
    0,
    1.05
)

plt.grid(
    axis="y"
)

plt.show()


# ============================================================
# 20. ACCURACY COMPARISON
# ============================================================

plt.figure(figsize=(9, 5))

plt.bar(
    results_df["Dataset"],
    results_df["Accuracy"]
)

plt.xlabel("Dataset Configuration")
plt.ylabel("Test Accuracy")

plt.title(
    "Accuracy Comparison - IR + Audio"
)

plt.ylim(
    0,
    1.05
)

plt.grid(
    axis="y"
)

plt.show()


# ============================================================
# 21. SAVE RESULTS
# ============================================================

results_df.to_csv(
    "wildlife_multimodal_results.csv",
    index=False
)

print("\n============================================")
print("IMPLEMENTATION COMPLETE")
print("============================================")

print(
    "\nResults saved to:"
    " wildlife_multimodal_results.csv"
)

print(
    "\nImportant:"
    " This notebook uses synthetic IR/audio signals."
)

print(
    "The results will NOT reproduce the paper's"
    " reported AUC of 0.990 without the actual"
    " dataset and exact preprocessing."
)